# [7.2] Feature Verbalizers - Exercises

Feature verbalizers are hypotheses about activations. In this notebook, you will build the loop that keeps those hypotheses honest:

```text
examples -> hypothesis -> held-out predictions -> counterexamples -> revision -> intervention control
```

The running idea is simple: if a direction looks like it activates on a pattern, your explanation should predict held-out examples and should predict what happens when we add that direction back into the model.

<details>
<summary>Help - how this differs from a classifier</summary>

A classifier can silently exploit whatever feature gives good accuracy. A verbalizer has to expose a compact human-readable rule, then let you attack that rule with contrastive examples and interventions.

</details>

<details>
<summary>Expected output</summary>

By the end, the committed report should show held-out accuracy `1.00`, contrastive accuracy `1.00`, score separation about `6.64`, and a target-direction intervention delta about `+0.77`.

</details>


In [ ]:
import json
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter7_activation_to_language"
section = "part2_feature_verbalizers"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_feature_verbalizers.tests as tests
import part2_feature_verbalizers.utils as utils

ExampleKind = Literal["top", "bottom", "random", "contrastive"]
TOKEN_RE = re.compile(r"[a-zA-Z]+")
DEFAULT_STOPWORDS = {
    "a", "an", "and", "at", "beside", "in", "near", "of", "on", "over", "the", "to",
}
MAIN = __name__ == "__main__"


@dataclass(frozen=True)
class VerbalizerExample:
    text: str
    score: float
    label: bool
    kind: ExampleKind


@dataclass(frozen=True)
class VerbalizerExampleSet:
    top: tuple[VerbalizerExample, ...]
    bottom: tuple[VerbalizerExample, ...]
    random: tuple[VerbalizerExample, ...]
    contrastive: tuple[VerbalizerExample, ...]


@dataclass(frozen=True)
class ExplanationPredictionReport:
    accuracy: float
    baseline_accuracy: float
    contrastive_accuracy: float
    passes_baseline: bool
    survives_contrastive: bool


@dataclass(frozen=True)
class CounterexampleReport:
    num_counterexamples: int
    counterexamples: tuple[str, ...]


@dataclass(frozen=True)
class InterventionPredictionReport:
    predicted_direction: Literal["increase", "decrease"]
    observed_delta: float
    matches_prediction: bool


@dataclass(frozen=True)
class ExplanationBrevityReport:
    explanation_word_count: int
    examples_word_count: int
    shorter_than_examples: bool


## A tiny inspection table

The toy examples below are deliberately small so the tests can be exact. The same functions are used in the pinned `gelu-1l` preflight on safe generated prompts.

<details>
<summary>Help - what should you look for?</summary>

Start by looking at both extremes and the examples near the threshold. If your hypothesis only explains the top examples, it is probably too broad.

</details>


In [ ]:
toy_texts = ["alpha code", "beta def", "plain story", "quiet notes"]
toy_scores = t.tensor([0.9, 0.8, 0.2, 0.1])
toy_labels = t.tensor([1, 1, 0, 0], dtype=t.bool)
inspection_rows = [
    {"text": text, "score": float(score), "label": bool(label)}
    for text, score, label in zip(toy_texts, toy_scores, toy_labels, strict=True)
]
inspection_rows


## Example Collection

Implement the function that gathers top, bottom, random, and contrastive examples.

<details>
<summary>Expected output</summary>

Top examples: `alpha code`, `beta def`. Bottom examples: `quiet notes`, `plain story`. Contrastive examples: `beta def`, `plain story`.

</details>

<details>
<summary>Help - why contrastive examples?</summary>

Near-threshold examples are where a verbal explanation is most likely to overreach. They are the first cheap adversarial examples for your story.

</details>

<details>
<summary>Solution</summary>

Flatten scores and labels, validate matching shapes, use `topk` for the extremes, seeded `randperm` for random examples, and `abs(scores - threshold).argsort()` for contrastives.

</details>

Common bug: forgetting to preserve the example kind, which makes later debugging harder.


In [ ]:
def gather_verbalizer_examples(
    texts: list[str],
    scores: t.Tensor,
    labels: t.Tensor,
    *,
    k: int = 3,
    threshold: float | None = None,
    seed: int = 0,
) -> VerbalizerExampleSet:
    raise NotImplementedError()


tests.test_gather_verbalizer_examples_covers_top_bottom_random_contrastive(
    gather_verbalizer_examples,
)
tests.test_gather_verbalizer_examples_rejects_bad_shapes_and_k(gather_verbalizer_examples)


## Explanation Predictions

Now convert a short explanation into predictions and score it against a baseline.

<details>
<summary>Expected output</summary>

For `texts = ["write code", "plain story", "def fn", "quiet notes"]` and terms `code`, `def`, predictions should be `[True, False, True, False]`; accuracy should be `1.0`; baseline accuracy should be `0.5`.

</details>

<details>
<summary>Help - avoid substring traps</summary>

`cat` should match `cat sat`, not `catalog`. Tokenize the text before comparing explanation terms.

</details>

<details>
<summary>Solution</summary>

Use `TOKEN_RE.findall(text.lower())`, convert the result to a set, and check whether any normalized explanation term appears as a whole token. The report should flatten tensors, validate all shapes match, and compute ordinary accuracies.

</details>

Common bug: evaluating the examples you used to write the hypothesis and calling that held-out accuracy.


In [ ]:
def keyword_explanation_predictions(
    texts: list[str],
    explanation_terms: list[str],
) -> t.Tensor:
    raise NotImplementedError()


def explanation_prediction_report(
    predictions: t.Tensor,
    labels: t.Tensor,
    baseline_predictions: t.Tensor,
    contrastive_mask: t.Tensor,
) -> ExplanationPredictionReport:
    raise NotImplementedError()


tests.test_keyword_predictions_and_explanation_report_use_baseline_and_contrastives(
    keyword_explanation_predictions,
    explanation_prediction_report,
)
tests.test_keyword_predictions_reject_empty_explanation_terms(keyword_explanation_predictions)
tests.test_keyword_predictions_do_not_match_substrings(keyword_explanation_predictions)
tests.test_explanation_prediction_report_rejects_shape_mismatch(explanation_prediction_report)


## Learn Terms From Train Only

The real report learns the verbalizer terms from training positives only. This is a small but concrete leakage guard.

<details>
<summary>Expected output</summary>

Learned terms should come from positive training examples. Negative-only words such as `flew` and `floated` should not appear.

</details>

<details>
<summary>Help - why not inspect held-out first?</summary>

If you read held-out examples before choosing the explanation, held-out accuracy becomes a post-hoc fit. Keep term selection on the train split.

</details>

<details>
<summary>Solution</summary>

Count unique non-stopword tokens per example, store negative and positive counts separately, and sort by `(pos - neg, pos, -neg, token)`.

</details>

Common bug: picking frequent words without subtracting negative counts.


In [ ]:
def learn_verbalizer_terms(
    texts: list[str],
    labels: t.Tensor,
    *,
    top_k: int = 5,
    stopwords: set[str] | None = None,
) -> list[str]:
    raise NotImplementedError()


tests.test_learned_verbalizer_terms_do_not_use_heldout_only_words(learn_verbalizer_terms)
tests.test_learned_verbalizer_terms_reject_bad_inputs(learn_verbalizer_terms)


## Counterexamples And Revision

A good verbalizer loop should return the examples that break the explanation.

<details>
<summary>Expected output</summary>

`plain story` should be the single counterexample, and the revised explanation should append `Revision: Exclude ordinary stories.`

</details>

<details>
<summary>Help - false positives vs false negatives</summary>

False positives show your explanation is too broad. False negatives show it missed part of the mechanism. Keep both visible.

</details>

<details>
<summary>Solution</summary>

Find indices where `predictions != labels`, keep the first `max_examples`, and append a revision note only when counterexamples exist.

</details>

Common bug: revising the explanation without preserving the failed examples that caused the revision.


In [ ]:
def find_counterexamples(
    texts: list[str],
    predictions: t.Tensor,
    labels: t.Tensor,
    *,
    max_examples: int = 3,
) -> CounterexampleReport:
    raise NotImplementedError()


def revise_explanation(
    explanation: str,
    counterexamples: tuple[str, ...],
    *,
    revision_note: str,
) -> str:
    raise NotImplementedError()


tests.test_counterexamples_and_revision_are_grounded_in_failures(
    find_counterexamples,
    revise_explanation,
)
tests.test_counterexamples_reject_bad_inputs(find_counterexamples)


## Intervention Direction And Brevity

Held-out predictions are still correlational. The intervention report asks whether changing the activation moves the predicted score in the direction your explanation says it should.

<details>
<summary>Expected output</summary>

The intervention delta should be `0.4` and should match predicted direction `increase`. The brevity report should count `4` explanation words and `8` example words.

</details>

<details>
<summary>Help - what does the intervention add?</summary>

It checks sign, not just magnitude. If the explanation predicts an increase and the score decreases, the explanation failed even if the absolute change is large.

</details>

<details>
<summary>Solution</summary>

Use `intervened_scores.mean() - baseline_scores.mean()` and compare the sign with the requested direction. For brevity, compare word counts against the examples-only baseline.

</details>

Common bug: treating brevity as sufficient evidence. It only matters after prediction and intervention checks pass.


In [ ]:
def intervention_prediction_report(
    baseline_scores: t.Tensor,
    intervened_scores: t.Tensor,
    *,
    predicted_direction: Literal["increase", "decrease"],
) -> InterventionPredictionReport:
    raise NotImplementedError()


def explanation_brevity_report(
    explanation: str,
    examples: list[str] | tuple[str, ...],
) -> ExplanationBrevityReport:
    raise NotImplementedError()


tests.test_intervention_prediction_checks_signed_direction(intervention_prediction_report)
tests.test_intervention_prediction_rejects_invalid_direction(intervention_prediction_report)
tests.test_explanation_brevity_compares_against_examples_only_baseline(
    explanation_brevity_report,
)


## Signature Result

The final result is loaded from the committed CUDA report. This is not a replacement for the exercises above; it is the real-model preflight that uses the same loop.

<details>
<summary>Expected output</summary>

The report should show `preflight_passed == True`, held-out accuracy `1.0`, baseline accuracy `0.5`, contrastive accuracy `1.0`, intervention delta about `0.77`, random-direction delta about `0.24`, and peak VRAM below `1 GB`.

</details>

<details>
<summary>Help - how to interpret this result</summary>

The result supports one narrow claim: the verbalizer loop works on a pinned `gelu-1l` residual direction. It does not prove broad SAE feature verbalization or API-LLM explanation quality.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    return json.loads((section_dir / "verification_report.json").read_text())


report = _load_committed_gpu_report()
gpu = report["metrics"]["gpu_test"]
signature_rows = [
    ("explanation", gpu["explanation"]),
    ("held-out accuracy", gpu["prediction_accuracy"]),
    ("baseline accuracy", gpu["baseline_accuracy"]),
    ("contrastive accuracy", gpu["contrastive_accuracy"]),
    ("score separation", gpu["score_separation"]),
    ("intervention delta", gpu["intervention_delta"]),
    ("random direction delta", gpu["random_direction_intervention_delta"]),
    ("counterexamples", gpu["num_counterexamples"]),
    ("train/held-out overlap", gpu["train_heldout_overlap_count"]),
    ("peak VRAM GB", gpu["peak_vram_gb"]),
]
signature_rows


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(["negative", "positive"], [gpu["negative_mean_score"], gpu["positive_mean_score"]], color=["#dc2626", "#16a34a"])
axes[0].axhline(0, color="#475569", linewidth=1)
axes[0].set_title("Projection scores")
axes[0].set_ylabel("mean score")
axes[1].bar(["target", "random"], [gpu["intervention_delta"], gpu["random_direction_intervention_delta"]], color=["#0891b2", "#94a3b8"])
axes[1].axhline(0.5, color="#64748b", linestyle="--", linewidth=1)
axes[1].set_title("Intervention deltas")
fig.tight_layout()
plt.show()


## Full Verification Contract

The notebook exposes standard extension entry points so the same section can be run as a course exercise and as a verification artifact.

<details>
<summary>Expected output</summary>

`run_gpu_test(max_vram_gb=24.0)` should return the accepted report metrics, and `run_full_experiment` should call the same path.

</details>

<details>
<summary>Help - why read the committed report here?</summary>

The exercise notebook should be runnable on CPU while still showing the real CUDA result. The actual CUDA regeneration is done by `scripts/run_extension_verification_reports.py --section 7.2`.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return _load_committed_gpu_report()["metrics"]["notebook_contract"]


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = _load_committed_gpu_report()
    gpu_report = report["metrics"]["gpu_test"]
    assert gpu_report["peak_vram_gb"] <= max_vram_gb
    return gpu_report


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_gpu_test(max_vram_gb=24.0)


## Limitations

This is a GT-1 residual-direction verbalizer preflight. It validates one pinned `gelu-1l` direction on small safe generated prompt sets. It does not claim API-LLM verbalization, SAE feature explanation, broad OOD generalization, or robust causal control over generations.

## Further Research

Try a different direction, deliberately write an overbroad explanation, add a negative-direction intervention, or replace the keyword parser with a local LLM while keeping the same held-out and intervention gates.
